**Intelligent Vision Systems · Lab 7 · Reading & Implementing Research Papers**
Companion to **Lecture 7 — Reading & Implementing Research Papers**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/03-transformer-reverse-sequence.ipynb) &nbsp; [Lecture 7 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/07_Paper_Implementation.html#/transformer-build-order) &nbsp; [All Lab 7 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-7-papers) &nbsp; [Setup guide: Colab & Cursor](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/09_paper_implementation_setup.html)

# Lab 7C · The Transformer, Block by Block: Reverse a Sequence

| ⏱ Time | 💻 Runs on | 🎯 You will build |
|---|---|---|
| 90 min (training ≈ 1 min on CPU) | Google Colab (CPU is enough, GPU is faster) or Cursor / VS Code on your laptop | positional encoding, LayerNorm, the feed-forward network, encoder and decoder layers, the full encoder–decoder Transformer, and a model that reverses any sequence of digits |

**How to work.** Run the cells from top to bottom with **Shift + Enter**. Cells marked **🖐 Your turn** have a `# TODO`:
write a few lines, run the cell, then run the **✅ check** below it. A 💡 means "not yet": read the hint, fix, run again.
Stuck for more than 10 minutes? Open the **Hint** boxes, look at the matching lecture slide, or ask the AI assistant to
*explain* (not write) the step. 

📄 **Paper:** Vaswani et al., *Attention Is All You Need* · [arXiv:1706.03762](https://arxiv.org/abs/1706.03762) · read
**§3.1, §3.3–3.5 and Figure 1**. Optional companion: [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) (Harvard NLP).

We build Figure 1 in the same order as the lecture, **smallest block first, testing each one before using it**:

| Step | Block | Paper | Check |
|---|---|---|---|
| 1 | Positional encoding | §3.5 | values = formula |
| 2 | Layer normalisation | §3.1 ("Add & Norm") | = `nn.LayerNorm` |
| 3 | Position-wise feed-forward | §3.3, Eq. 2 | shape + parameter count |
| 4 | Encoder layer | §3.1 | shape, padding mask, parameters |
| 5 | Decoder layer | §3.1 | causal mask, uses the encoder |
| 6 | Full Transformer | Figure 1 | shapes, no peeking at the future |
| 7 | Train: reverse `3 7 1 9` → `9 1 7 3` | — | ≥ 95 % of sequences perfect |

Why reversing? It is tiny (trains in a minute on a CPU) but impossible without the two key ideas of the paper:
**attention** (look anywhere in the input) and **positions** (know *where* each digit was).

In [ ]:
# Setup: run this first (Colab already has PyTorch; on your laptop see requirements.txt).
# !pip install -q torch torchvision matplotlib scikit-learn opencv-python

import os, urllib.request
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_7"
for f in ["lab7_checks.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{RAW}/{f}", f)
print("lab files ready")

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import lab7_checks as C          # auto-checks + helpers for this lab

C.seed_everything(0)
device = C.get_device()
print("PyTorch", torch.__version__, "· device:", device)

In [ ]:
# Toolbox from Lab 7B (our solution; paste yours instead if you like)
def attention(q, k, v, mask=None):
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    weights = scores.softmax(dim=-1)
    return weights @ v, weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        self.h, self.d_k = n_heads, d_model // n_heads
        self.w_q, self.w_k = nn.Linear(d_model, d_model), nn.Linear(d_model, d_model)
        self.w_v, self.w_o = nn.Linear(d_model, d_model), nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
        self.attn = None

    def forward(self, q, k, v, mask=None):
        B, Lq, D = q.shape
        Lk = k.size(1)
        q = self.w_q(q).view(B, Lq, self.h, self.d_k).transpose(1, 2)
        k = self.w_k(k).view(B, Lk, self.h, self.d_k).transpose(1, 2)
        v = self.w_v(v).view(B, Lk, self.h, self.d_k).transpose(1, 2)
        out, self.attn = attention(q, k, v, mask)
        return self.w_o(self.drop(out.transpose(1, 2).reshape(B, Lq, D)))

print("attention + MultiHeadAttention ready")

## 1 · Positional encoding (§3.5)

Attention treats the input as a **set**: shuffle the words and every output is shuffled the same way. The model needs to
be told the order. The paper adds a fixed pattern of sines and cosines to every embedding:

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right),\qquad PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

Even dimensions get sin, odd dimensions cos; low dimensions wiggle fast, high dimensions slowly (like the hands of a clock).

### 🖐 Your turn 1: `positional_encoding(max_len, d_model)` → (max_len, d_model)
<details><summary><b>Hint 1</b></summary>

`pos = torch.arange(max_len).unsqueeze(1).float()` has shape (max_len, 1).

</details>
<details><summary><b>Hint 2</b></summary>

`two_i = torch.arange(0, d_model, 2).float()` are the even indices 0, 2, 4 …; `div = 10000 ** (two_i / d_model)`.

</details>
<details><summary><b>Hint 3</b></summary>

`pe[:, 0::2] = torch.sin(pos / div)` and `pe[:, 1::2] = torch.cos(pos / div)` (broadcasting does the rest).

</details>

### 📄 ↔ 💻 Paper ↔ Code: the sinusoidal positional encoding → `positional_encoding()`

<table>
<tr><th width="50%" align="left">📄 Paper · Vaswani et al. 2017, §3.5</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_pe.png" width="100%" alt="pc_t_pe.png"><br></td>
<td width="50%" valign="top">
<b>💻 Your implementation plan: one line per box / term of the paper</b>
<pre>def positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    # pos  = 0 .. max_len-1 as a column            (max_len, 1)
    # 2i   = 0, 2, 4, ..., d_model-2               (d_model/2,)
    # div  = 10000 ** (2i / d_model)
    # even columns pe[:, 0::2] = sin(pos / div)    PE(pos, 2i)
    # odd  columns pe[:, 1::2] = cos(pos / div)    PE(pos, 2i+1)
    return pe</pre>
<b>Shapes:</b> pos (L, 1) / div (d/2,) broadcasts to (L, d/2); the result (L, d<sub>model</sub>) is added to every sentence
</td></tr>
</table>

In [ ]:
def positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    # TODO
    return pe

In [ ]:
C.check_positional_encoding(positional_encoding)
pe = positional_encoding(60, 64)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].imshow(pe.T, aspect="auto", cmap="RdBu"); ax[0].set_xlabel("position"); ax[0].set_ylabel("dimension"); ax[0].set_title("PE matrix (60 positions × 64 dims)")
for d in [0, 2, 10, 30]:
    ax[1].plot(pe[:, d], label=f"dim {d}")
ax[1].set_xlabel("position"); ax[1].legend(); ax[1].set_title("low dims wiggle fast, high dims slowly")
plt.tight_layout(); plt.show()

## 2 · Layer normalisation (the "Norm" in "Add & Norm")

For every token separately: subtract the mean of its $d_{model}$ features, divide by their standard deviation, then
scale and shift with learned $\gamma$ (ones) and $\beta$ (zeros):
$\;\mathrm{LN}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$.

### 🖐 Your turn 2: `LayerNorm`
<details><summary><b>Hint 1</b></summary>

Mean and variance over the LAST dimension: `x.mean(-1, keepdim=True)`, `x.var(-1, keepdim=True, unbiased=False)`.

</details>
<details><summary><b>Hint 2</b></summary>

`gamma` and `beta` must be `nn.Parameter`s so the optimizer can learn them.

</details>

### 📄 ↔ 💻 Paper ↔ Code: "LayerNorm(x + Sublayer(x))" → `LayerNorm` and Add & Norm

<table>
<tr><th width="50%" align="left">📄 Paper · Vaswani et al. 2017, §3.1</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_addnorm.png" width="100%" alt="pc_t_addnorm.png"><br></td>
<td width="50%" valign="top">
<b>💻 Your implementation plan: one line per box / term of the paper</b>
<pre>class LayerNorm(nn.Module):
    # learnable gamma = ones(d), beta = zeros(d)
    def forward(self, x):                      # x (B, L, d)
        # mean and variance over the LAST dim (features of ONE token)
        # (x - mean) / sqrt(var + eps) * gamma + beta    (unbiased=False)
&nbsp;
# &quot;the output of each sub-layer is LayerNorm(x + Sublayer(x))&quot;
x = norm(x + dropout(sublayer(x)))             # = one &quot;Add &amp; Norm&quot; box</pre>
</td></tr>
</table>

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        # TODO: self.gamma, self.beta

    def forward(self, x):
        # TODO
        return x

In [ ]:
C.check_layernorm(LayerNorm)

## 3 · Position-wise feed-forward network (§3.3, Eq. 2)

$$\mathrm{FFN}(x) = \max(0,\, xW_1 + b_1)\,W_2 + b_2$$

Two linear layers with a ReLU between them, applied to **each position separately** (an `nn.Linear` on a (B, L, D)
tensor does exactly that). The paper uses $d_{model}=512$, $d_{ff}=2048$.

### 🖐 Your turn 3: `FeedForward(d_model, d_ff, dropout)`

### 📄 ↔ 💻 Paper ↔ Code: Eq. (2) → `FeedForward`

<table>
<tr><th width="50%" align="left">📄 Paper · Vaswani et al. 2017, §3.3, Eq. (2)</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_ffn.png" width="100%" alt="pc_t_ffn.png"><br></td>
<td width="50%" valign="top">
<b>💻 Your implementation plan: one line per box / term of the paper</b>
<pre>class FeedForward(nn.Module):
    # W1, b1 = nn.Linear(d_model, d_ff)     512 -&gt; 2048
    # W2, b2 = nn.Linear(d_ff, d_model)     2048 -&gt; 512
    def forward(self, x):                   # (B, L, d_model)
        # max(0, x W1 + b1) W2 + b2  -&gt;  lin2( relu( lin1(x) ) )
        # nn.Linear on (B, L, d) acts on each position separately</pre>
<b>Shapes:</b> (B, L, 512) → (B, L, 2048) → ReLU → (B, L, 512)
</td></tr>
</table>

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        # TODO: two nn.Linear layers and an nn.Dropout

    def forward(self, x):
        # TODO
        return x

In [ ]:
C.check_ffn(FeedForward)

## 4 · Encoder layer (§3.1, Figure 1 left)

> *"Each layer has two sub-layers … We employ a residual connection around each of the two sub-layers, followed by
> layer normalization. That is, the output of each sub-layer is LayerNorm(x + Sublayer(x))."*

```
x ─► self-attention(x, x, x, mask) ─► dropout ─► (+ x) ─► LayerNorm ─► FFN ─► dropout ─► (+) ─► LayerNorm ─► out
└──────────────── residual ─────────────────────┘                   └──────── residual ───────┘
```

### 🖐 Your turn 4: `EncoderLayer.forward`
<details><summary><b>Hint 1</b></summary>

`x = self.norm1(x + self.drop(self.self_attn(x, x, x, src_mask)))`

</details>
<details><summary><b>Hint 2</b></summary>

Then the same pattern with `self.ffn` and `self.norm2`.

</details>

### 📄 ↔ 💻 Paper ↔ Code: Figure 1 (left) → `EncoderLayer.forward`

<table>
<tr><th width="50%" align="left">📄 Paper · Vaswani et al. 2017, §3.1, Figure 1</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/p_transformer_fig1.png" width="100%" alt="p_transformer_fig1.png"><br></td>
<td width="50%" valign="top">
<b>💻 Your implementation plan: one line per box / term of the paper</b>
<pre>class EncoderLayer(nn.Module):
    def forward(self, x, src_mask):
        # Figure 1, left column, bottom to top:
        # [Multi-Head Attention]  self-attention: q = k = v = x, src_mask
        # [Add &amp; Norm]            x = norm1(x + drop(attention))
        # [Feed Forward]          ffn(x)
        # [Add &amp; Norm]            x = norm2(x + drop(ffn))
        return x                  # (B, L_src, d_model): shape never changes</pre>
</td></tr>
</table>

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1, self.norm2 = LayerNorm(d_model), LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        # TODO: two sub-layers, each LayerNorm(x + Dropout(Sublayer(x)))
        return x

In [ ]:
C.check_encoder_layer(EncoderLayer)

## 5 · Decoder layer (§3.1, Figure 1 right)

Three sub-layers, each with Add & Norm:
1. **masked** self-attention over the output so far (`tgt_mask` = causal mask),
2. **cross-attention**: queries from the decoder, keys and values from the encoder output (`memory`),
3. the feed-forward network.

### 🖐 Your turn 5: `DecoderLayer.forward`
<details><summary><b>Hint 1</b></summary>

Cross-attention: `self.cross_attn(y, memory, memory, src_mask)`: the first argument gives the queries.

</details>

### 📄 ↔ 💻 Paper ↔ Code: Figure 1 (right) → `DecoderLayer.forward`

<table>
<tr><th width="50%" align="left">📄 Paper · Vaswani et al. 2017, §3.1, §3.2.3, Figure 1</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/p_transformer_fig1.png" width="100%" alt="p_transformer_fig1.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_mask.png" width="100%" alt="pc_t_mask.png"><br></td>
<td width="50%" valign="top">
<b>💻 Your implementation plan: one line per box / term of the paper</b>
<pre>def forward(self, y, memory, src_mask, tgt_mask):
    # Figure 1, right column, bottom to top:
    # [Masked Multi-Head Attention]  q = k = v = y, with tgt_mask (causal)
    # [Add &amp; Norm]
    # [Multi-Head Attention]         cross-attention: q = y, k = v = memory
    # [Add &amp; Norm]
    # [Feed Forward] + [Add &amp; Norm]
    return y                          # (B, L_tgt, d_model)</pre>
<b>Shapes:</b> cross-attention weights: (B, h, L<sub>tgt</sub>, L<sub>src</sub>): every output token looks at every input token
</td></tr>
</table>

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1, self.norm2, self.norm3 = LayerNorm(d_model), LayerNorm(d_model), LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, y, memory, src_mask, tgt_mask):
        # TODO: masked self-attention, cross-attention, FFN (each with Add & Norm)
        return y

In [ ]:
C.check_decoder_layer(DecoderLayer)

## 6 · The full Transformer (Figure 1)

Embedding × $\sqrt{d_{model}}$ (§3.4) + positional encoding → N encoder layers → N decoder layers → Linear → scores
over the vocabulary (the softmax is inside the loss). Two masks:
* **source padding mask** (B, 1, 1, L_src): `True` where the source token is not `PAD` (we have no padding here, but real data does);
* **target causal mask** (1, 1, L_tgt, L_tgt): from your `causal_mask`.

### 🖐 Your turn 6: build the two masks in `encode` and `decode`
<details><summary><b>Hint 1</b></summary>

`src_mask = (src != self.pad).view(src.size(0), 1, 1, src.size(1))`

</details>
<details><summary><b>Hint 2</b></summary>

`tgt_mask = torch.tril(torch.ones(L, L, dtype=torch.bool, device=tgt.device)).view(1, 1, L, L)`

</details>

### 📄 ↔ 💻 Paper ↔ Code: embeddings, §3.4, and the whole of Figure 1

<table>
<tr><th width="50%" align="left">📄 Paper · Vaswani et al. 2017, §3.4, §3.5, Figure 1</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_emb.png" width="100%" alt="pc_t_emb.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/p_transformer_fig1.png" width="100%" alt="p_transformer_fig1.png"><br></td>
<td width="50%" valign="top">
<b>💻 Code</b>
<pre># §3.4 &quot;we multiply those weights by sqrt(d_model)&quot; + §3.5 positional encoding
def embed(self, emb, tokens):
    return self.drop(emb(tokens) * math.sqrt(self.d_model)
                     + self.pe[: tokens.size(1)])
&nbsp;
memory = self.encode(src)                       # N x EncoderLayer  (B, L_src, d)
logits = self.out(self.decode(tgt_in, memory))  # N x DecoderLayer -&gt; Linear
# [Softmax] is inside F.cross_entropy(logits, targets)</pre>
</td></tr>
</table>

> **Paper vs lab:** the paper *shares* one weight matrix between both embeddings and the output Linear (§3.4). The lab keeps them separate (simpler, and the vocabularies are tiny). Write such differences in your report.

In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=64, n_heads=4, n_layers=2, d_ff=128, dropout=0.0, max_len=100, pad=0):
        super().__init__()
        self.d_model, self.pad = d_model, pad
        self.src_emb = nn.Embedding(src_vocab, d_model)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model)
        self.register_buffer("pe", positional_encoding(max_len, d_model))   # saved with the model, not learned
        self.encoder = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.decoder = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.out = nn.Linear(d_model, tgt_vocab)
        self.drop = nn.Dropout(dropout)

    def embed(self, emb, tokens):
        return self.drop(emb(tokens) * math.sqrt(self.d_model) + self.pe[: tokens.size(1)])

    def encode(self, src):
        src_mask = None   # TODO: (B, 1, 1, L_src) bool, True where src != pad
        x = self.embed(self.src_emb, src)
        for layer in self.encoder:
            x = layer(x, src_mask)
        return x, src_mask

    def decode(self, tgt, memory, src_mask):
        L = tgt.size(1)
        tgt_mask = None   # TODO: (1, 1, L, L) causal mask
        y = self.embed(self.tgt_emb, tgt)
        for layer in self.decoder:
            y = layer(y, memory, src_mask, tgt_mask)
        return self.out(y)

    def forward(self, src, tgt):
        memory, src_mask = self.encode(src)
        return self.decode(tgt, memory, src_mask)

In [ ]:
VOCAB = 13          # PAD = 0, BOS = 1, EOS = 2, digits 0–9 are tokens 3–12
C.check_transformer(Transformer(VOCAB, VOCAB))
print("The paper's base model (d_model=512, 8 heads, 6+6 layers, d_ff=2048, 37k-word vocabulary) has ~65 M parameters;")
print("ours has", f"{C.count_params(Transformer(VOCAB, VOCAB)):,}", "— same code, smaller numbers.")

## 7 · The data: reverse a sequence

`C.make_reverse_batch` makes random digit sequences. The decoder is trained with **teacher forcing**: its input is the
correct answer shifted right by one (starting with `BOS`), and it must predict the next token at every position.

```
src      :      3  7  1  9          (we show digits; tokens are digit + 3)
tgt_in   : BOS  9  1  7  3          decoder input  (shifted right)
tgt_out  :  9   1  7  3 EOS         what it must predict
```

In [ ]:
src, tgt_in, tgt_out = C.make_reverse_batch(batch_size=3, seq_len=8)
show = lambda t: " ".join({0: "PAD", 1: "BOS", 2: "EOS"}.get(int(v), str(int(v) - 3)) for v in t)
for i in range(3):
    print("src    :", show(src[i])); print("tgt_in :", show(tgt_in[i])); print("tgt_out:", show(tgt_out[i])); print()

## 8 · Train (≈ 1 minute on a CPU, seconds on a GPU)

The paper uses Adam with β = (0.9, 0.98) and a warm-up schedule (Eq. 3); for this small model a constant learning
rate of 2e-3 is enough. The loss is cross-entropy between the scores (B, L, V) and `tgt_out` (B, L).

In [ ]:
SEQ_LEN, STEPS = 8, 1000
C.seed_everything(0)
model = Transformer(VOCAB, VOCAB, d_model=64, n_heads=4, n_layers=2, d_ff=128, dropout=0.0).to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3, betas=(0.9, 0.98))
losses, t0 = [], time.time()
model.train()
for step in range(1, STEPS + 1):
    src, tgt_in, tgt_out = C.make_reverse_batch(64, SEQ_LEN, device=device)
    logits = model(src, tgt_in)                                   # (B, L+1, VOCAB)
    loss = F.cross_entropy(logits.reshape(-1, VOCAB), tgt_out.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if step % 200 == 0:
        print(f"step {step:4d} · loss {loss.item():.4f} · {time.time() - t0:.0f} s")
plt.figure(figsize=(5, 2.8)); plt.plot(losses); plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss"); plt.title("training loss"); plt.show()

## 9 · Inference: greedy decoding

At test time there is no answer to feed in. Start with `BOS`, predict one token, append it, predict the next… (this is
exactly how ChatGPT writes, one token at a time).

In [ ]:
@torch.no_grad()
def greedy_decode(model, src, max_len):
    model.eval()
    memory, src_mask = model.encode(src)
    y = torch.full((src.size(0), 1), 1, dtype=torch.long, device=src.device)    # BOS
    for _ in range(max_len):
        next_tok = model.decode(y, memory, src_mask)[:, -1].argmax(-1, keepdim=True)
        y = torch.cat([y, next_tok], dim=1)
    return y[:, 1:]

src, _, tgt_out = C.make_reverse_batch(1000, SEQ_LEN, device=device)
pred = greedy_decode(model, src, SEQ_LEN + 1)
acc = (pred == tgt_out).all(dim=1).float().mean().item()
for i in range(3):
    print("input:", show(src[i]), " →  output:", show(pred[i]))
C.check_reverse_accuracy(acc)

## 10 · Look inside: the cross-attention learned to read backwards

For output position *i* the decoder should look at input position *L − 1 − i*: an **anti-diagonal**. Nobody told the
model; it discovered it from data.

In [ ]:
x = torch.tensor([[5, 2, 8, 1, 9, 4, 7, 3]], device=device) + 3
out = greedy_decode(model, x, SEQ_LEN + 1)
with torch.no_grad():                                                     # one teacher-forced pass to record the weights
    model.decode(torch.cat([torch.ones(1, 1, dtype=torch.long, device=device), out[:, :-1]], 1), *model.encode(x))
A = model.decoder[-1].cross_attn.attn[0].mean(0).cpu()                  # average over heads: (L_tgt, L_src)
plt.figure(figsize=(4.5, 4)); plt.imshow(A, cmap="Blues")
tok = lambda v: {0: "PAD", 1: "BOS", 2: "EOS"}.get(int(v), str(int(v) - 3))       # one token -> text
plt.xticks(range(8), [tok(t) for t in x[0]]); plt.yticks(range(9), [tok(t) for t in out[0]])
plt.xlabel("input token"); plt.ylabel("output token"); plt.title("cross-attention (last decoder layer)"); plt.colorbar(); plt.show()

## ✍️ Your turn: small experiments (this is what "research" looks like)
Change **one thing at a time**, re-run sections 8–10, and write down the accuracy in a table.

1. **No positional encoding:** in `embed`, remove `+ self.pe[...]`. What accuracy do you get, and why? (Hint: §3.5 and Lecture slide "Why positions?")
2. **One head instead of four** (`n_heads=1`). Faster or slower to learn?
3. **Sort instead of reverse:** change `make_reverse_batch` so the target is `src.sort(1).values`. Does it still work? What does the cross-attention look like now?
4. **Longer sequences** (`SEQ_LEN = 16`): how many steps do you need?
5. ⭐ Implement the paper's learning-rate schedule (Eq. 3): $lrate = d_{model}^{-0.5}\cdot\min(step^{-0.5},\ step\cdot warmup^{-1.5})$ with `torch.optim.lr_scheduler.LambdaLR`.

**Next:** [Lab 7D · U-Net from the paper](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/04-unet-from-paper.ipynb)

### 📄 ↔ 💻 Paper ↔ Code: what the paper reports, and the experiments you can copy (Tables 2 and 3)

<table>
<tr><th width="50%" align="left">📄 Paper · Vaswani et al. 2017, Tables 2 and 3</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_table3.png" width="100%" alt="pc_t_table3.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_table2.png" width="100%" alt="pc_t_table2.png"><br></td>
<td width="50%" valign="top">
<b>💻 Code</b>
<pre># Table 3, rows (A): change ONLY the number of heads h.
# Same idea on our task (keep everything else fixed):
for h in [1, 2, 4, 8]:
    model = Transformer(VOCAB, VOCAB, d_model=64, n_heads=h,
                        n_layers=2, d_ff=128)
    print(h, C.count_params(model))   # identical for every h
    # train it as in section 8 and record the exact-sequence accuracy
&nbsp;
# Table 3, rows (C): change n_layers or d_model instead.
# Paper: 27.3 BLEU (base), 28.4 BLEU (big) on EN-DE after 12 h on 8 GPUs.
# You:   ~99 % reversed sequences after ~1 min on a CPU.</pre>
</td></tr>
</table>